⚠️ **IMPORTANT — Before you begin:** Make your own copy of this notebook!

In Google Colab: go to **File → Save a copy in Drive**, then rename it to something like `LastName_Lab9.ipynb` before doing any work. Working in the original shared notebook means your changes may be lost or overwritten.

# Lab 9: Deep Learning with Keras/TensorFlow — Fashion-MNIST

We've spent most of the last two weeks discussing neural networks and how to implement them.  We worked through an example of implementing a multilayer perceptron (MLP) using *Keras* and *TensorFlow* to analyze the famous [MNIST dataset](https://en.wikipedia.org/wiki/MNIST_database).  Keep in mind that dataset was quite simple and developed in the last 1980s, mostly before the era of deep learning. In fact, one the first contests to classify MNIST was won using a nearest-neighbor classifier with a custom similarity metric! As you saw in lecture, the MNIST dataset is so simple that even a very basic MLP can achieve over 98% accuracy on the test set. 

In this lab you will apply those techniques to a new (but in many ways similar) image classification dataset — **Fashion-MNIST** — and explore how network architecture choices affect performance.

Specifically, you will:

1. Learn about the Fashion-MNIST dataset and visualize sample images.
2. Build and evaluate a baseline MLP with two hidden layers, including early stopping.
3. Add **Dropout** regularization and measure its effect.
4. Replace dropout with a **bottleneck (narrow middle) layer** and see how compressing information flow changes accuracy.
5. Build a simple **Convolutional Neural Network (ConvNet/CNN)** and compare it to the MLP.

## Using a GPU in Google Colab

This lab is designed to run on **Google Colab**.  Because training neural networks involves a lot of matrix multiplications, it runs *much* faster on a GPU than on a CPU.  Now to be fair, the models we will be training in this lab are small enough that you could get away with using a CPU, but it will be much more efficient to use a GPU if it is available.  

Follow these steps to enable the free GPU before running any code cells:

1. In the Colab menu, click **Runtime → Change runtime type**.
2. Under *Hardware accelerator*, choose **T4 GPU** (or whichever GPU option is available).
3. Click **Save**.

> **Free-tier GPU limits:** Google Colab's free plan gives you access to a shared GPU (typically an NVIDIA T4 with ~15 GB of VRAM) for a limited number of hours per day.  Exactly how much time you get varies and is not publicly documented, but in practice you will generally have enough to complete this lab in a single session.  

If you run out, you will be prompted to wait or upgrade to Colab Pro.  For this lab, expect each model training run to take roughly **30 seconds** with the GPU enabled.  With the GPU disabled, training will likely take **2-3 minutes** per run, doable, but much less efficient.

Run the cell below to confirm that *TensorFlow* can see the GPU.

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

## Part A: The Fashion-MNIST Dataset

### Background

[**Fashion-MNIST**](https://en.wikipedia.org/wiki/Fashion-MNIST) was created by Zalando Research (a European e-commerce company) and released in 2017 as a drop-in replacement for the classic handwritten-digit MNIST benchmark.  The motivation was that MNIST had become too easy: many simple classifiers achieved over 99% accuracy, making it hard to distinguish between competing approaches.  Fashion-MNIST was designed to be harder while keeping exactly the same format so that code written for MNIST would work without modification.

The dataset consists of **70,000 grayscale images** (60,000 training + 10,000 test), each **28 × 28 pixels**.  Every image belongs to one of **10 clothing/accessory categories**:

| Label | Class | 
|-------|-------|
| 0 | T-shirt/top |
| 1 | Trouser |
| 2 | Pullover |
| 3 | Dress |
| 4 | Coat |
| 5 | Sandal |
| 6 | Shirt |
| 7 | Sneaker |
| 8 | Bag |
| 9 | Ankle boot |

State-of-the-art models achieve today around 96% accuracy on Fashion-MNIST, while simple MLPs typically land in the 87–91% range — a much more discriminating benchmark than the original MNIST.

### Step A.1 — Load and Inspect the Data

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Class names corresponding to label 0-9
CLASS_NAMES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

# Load the full Fashion-MNIST dataset (Keras downloads it automatically)
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = \
    keras.datasets.fashion_mnist.load_data()

print(f"Training images shape : {X_train_raw.shape}")
print(f"Training labels shape : {y_train_raw.shape}")
print(f"Test images shape     : {X_test_raw.shape}")
print(f"Test labels shape     : {y_test_raw.shape}")
print(f"Pixel value range     : {X_train_raw.min()} – {X_train_raw.max()}")

**Question 1:** How many features would be in the input layer of an MLP that processes these images?  How does that compare to the original MNIST dataset?

```
ANSWER HERE
```

### Step A.2 — Visualize Sample Images

Before building any model it is always a good idea to look at your data.  The cell below displays a random grid of images with their true class labels. **NOTE**: Since we are seeding the random number generator, you should see the same images and labels as shown below.  If you run the cell multiple times, you will get the same output each time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
nx = 10
ny = 10
sample_indices = np.random.choice(len(X_train_raw), nx*ny, replace=False)

fig, axes = plt.subplots(nx, ny, figsize=(8, 8))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(X_train_raw[idx], cmap='gray_r')
    ax.set_title(CLASS_NAMES[y_train_raw[idx]], fontsize=8)
    ax.axis('off')
plt.suptitle(f'{nx*ny} Sample Fashion-MNIST Images', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Question 2:** Looking at the sample images, answer the following questions:

- Similar to how the same digit can be written in many different ways, do you see a lot of variation in how the same clothing item is depicted?  Do you think the variation will be larger or smaller than the variation in how the same digit is written in MNIST?
- Which pairs of classes do you think will be *hardest* to tell apart, and why?  (Think about visual similarity.)

```
ANSWER HERE
```

### Step A.3 — Preprocess the Data

Neural networks train best when input values are small and centered near zero.  We normalize to bring the range to [0, 1] and then flatten the 28×28 images into 784-element vectors (required for a plain MLP).  

Normalize and flatten the training and test images using the code in the cell below like we did for MNIST in lecture. Remember NOT to use `normalize` from `keras.utils` since that is designed for normalizing vectors of features, not images.  Instead, we can simply divide the pixel values by 255 to scale them to the [0, 1] range.

In [ ]:
# Print range of raw values
print(f"Pixel value range (raw): {X_train_raw.min()} – {X_train_raw.max()}")

# TODO: Normalize the 'raw' training and testing data and reshape the
# 28x28 images into 784-element vectors.  DO NOT use the `normalize`
# function from `keras.utils` as it will try to set the maximum
# length of the 784-D vector to 1, which is not what we want.
# Instead, just divide the pixel values by 255.0 to normalize them to
# the [0, 1] range.
X_train = ...
X_test  = ...

# Print range of normalized values
print(f"Pixel value range (normalized): {X_train.min()} – {X_train.max()}")


Finish the following code to convert the integer class labels to **one-hot vectors** using `to_categorical`.

In [ ]:
from tensorflow.keras.utils import to_categorical

# TODO: Finish one-hot encoding the labels using `to_categorical`
y_train = ...
y_test  = ...

print("X_train shape:", X_train.shape, "  dtype:", X_train.dtype)
print("y_train shape:", y_train.shape)
print("Example one-hot label for first image:", y_train[0],
      "==> class", CLASS_NAMES[y_train_raw[0]])

## Part B: Baseline MLP — Two Hidden Layers

We will start with a straightforward MLP:

- **Input layer**: 784 nodes (one per pixel)
- **Hidden layer 1**: 256 nodes, ReLU activation
- **Hidden layer 2**: 128 nodes, ReLU activation
- **Output layer**: 10 nodes, softmax activation (one per class)

### Step B.1 — Build the Model

We will build an initial model using the Keras Sequential API. We are 
building it inside a function so that we can easily create multiple instances 
of the same architecture later on when we want to implement variations on it.

In [ ]:
from tensorflow.keras import layers

def build_mlp_baseline():
    """Baseline Two hidden-layer MLP."""

    # TODO: Build a simple MLP called `model` with the following
    # architecture:
    # - Input layer: 784 nodes (one per pixel)
    # - Hidden layer 1: 256 nodes, ReLU activation
    # - Hidden layer 2: 128 nodes, ReLU activation
    # - Output layer: 10 nodes, softmax activation (one per class)
    model = keras.Sequential(name='Baseline_MLP', layers=[
        ...
    ])

    return model

mlp_baseline = build_mlp_baseline()
mlp_baseline.summary()

**Question 3:** How many *total* trainable parameters does this model have?  Break it down layer by layer to verify the number shown in the summary.  (Remember that each Dense layer also has one bias weight per node.)

```
ANSWER HERE
```

### Step B.2 — Train the Model

Remember in TensorFlow/Keras, we must first `.compile()` the model to specify the loss function, optimizer, and metrics before we can call `fit()` to train it.  Finish the following code to compile and train the model.

**Validation Split:**

I am using a feature of the `fit()` method to hold out 10% of the training data for validation, which allows us to monitor the model's performance on unseen data during training.  This is important for detecting overfitting and for tuning hyperparameters later on.  We are adding this here so that we can plot the training and validation learning curves without touching the test set, which should only be used for the final evaluation after all model development is complete.

The training history will be stored in `history_baseline` so we can plot the learning curves after training.

In [ ]:
# TODO: Compile the model with the Adam optimizer, categorical cross-entropy 
# loss, and accuracy metric.  
mlp_baseline.compile(
    ...
)

# Train the model for 15 epochs with a batch size of 128, and hold out 
# 10% of the training data for validation.  Store the training history in 
# `history_baseline` so we can plot it later.
history_baseline = mlp_baseline.fit(
    X_train, y_train,
    batch_size=128,
    epochs=15,
    validation_split=0.1,   # hold out 10% of training data for validation
    verbose=1
)

### Step B.3 — Evaluate the Model

#### Learning Curves

We are going to define some functions for plotting the loss curve and examining metrics so that we can reuse them later when we train the other models. 

In [ ]:
def plot_history(history, title='Training History'):
    """Plot training and validation accuracy and loss."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    epochs = range(1, len(history.history['accuracy']) + 1)

    axes[0].plot(epochs, history.history['accuracy'],     'bo-', label='Train')
    axes[0].plot(epochs, history.history['val_accuracy'], 'r-',  label='Validation')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history.history['loss'],     'bo-', label='Train')
    axes[1].plot(epochs, history.history['val_loss'], 'r-',  label='Validation')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    fig.suptitle(title, fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history_baseline, 'Baseline MLP — Training History')

**Question 4:** Do the learning curves show any sign of **overfitting**?  How can you tell?

```
ANSWER HERE
```

#### Test-Set Metrics and Confusion Matrix

This function allows us to evaluate the model on the test set using both metrics and
a confusion matrix.  We can save the results to potentially redisplay later for 
comparison with other models.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

def evaluate_model(model, X_test, y_test_onehot, y_test_int, class_names,
                   title='', filename=None):
    """Print test-set accuracy and plot confusion matrix."""
    loss, acc = model.evaluate(X_test, y_test_onehot, verbose=0)
    report = f"\n{'='*50}\n"
    report += f"{title}\n"
    report += f"  Test loss    : {loss:.4f}\n"
    report += f"  Test accuracy: {acc:.4f} ({acc*100:.2f}%)\n"
    report += f"{'='*50}\n"

    # Generate integer predictions
    y_pred_int = np.argmax(model.predict(X_test, verbose=0), axis=1)
    class_rep = classification_report(y_test_int, y_pred_int,
                                      target_names=class_names)
    report += f"{class_rep}"

    # Confusion matrix
    cm = confusion_matrix(y_test_int, y_pred_int)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    fig, ax = plt.subplots(figsize=(9, 8))
    disp.plot(ax=ax, xticks_rotation=45, colorbar=True)
    ax.set_title(f'Confusion Matrix — {title}', fontsize=12)
    # Save the confusion matrix plot if a filename is provided, otherwise show it.
    if filename:
        fig.savefig(filename, bbox_inches='tight', facecolor='white', 
                    transparent=False)
        plt.close(fig)
    else:
        plt.show()

    return report

# Evaluate the baseline model on the test set and show the confusion matrix.
# Save the results to potentially redisplay later for comparison with
# the early-stopped model.
cm_file = 'baseline_mlp_confusion_matrix.png'
baseline_report = evaluate_model(mlp_baseline, X_test, y_test, y_test_raw,
               CLASS_NAMES, title='Baseline MLP (2 hidden layers)',
               filename=cm_file)

print(baseline_report)

# Display the confusion matrix for the baseline model
baseline_cm = plt.imread(cm_file)
plt.figure(figsize=(9, 8))
plt.imshow(baseline_cm)
plt.axis('off')
plt.tight_layout()
plt.show()

**Question 5:**

a. Look at the confusion matrix for the baseline model. Which two (or three) classes are most frequently confused with each other?  Does this match your prediction in Question 2?

b. Examine the code and explain how are the results of the evaluation and the confusion matrix being saved for later comparison?

```
ANSWER HERE
```

### Step B.4 — Reading the Learning Curves and Early Stopping

The learning curves you just plotted tell you more than just how well training went — they are your primary diagnostic tool for detecting overfitting and for deciding *which epoch's weights* you should actually use for predictions.

**How to read the curves:**

- As long as both training loss and validation loss are decreasing together, the model is still learning genuinely useful patterns — keep training.
- When the **training loss continues to drop** but the **validation loss levels off or starts to rise**, the model has begun to memorize the training data rather than generalize.  This is the onset of overfitting.  The best model weights are from the epoch just *before* the validation loss started to diverge upward.

**Early stopping** is a technique that automates this: you monitor the validation loss during training and stop (and save the best weights) automatically as soon as the loss stops improving.  In Keras this is done with a `callback`:

```python
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',   # watch validation loss
    patience=5,           # stop if no improvement for 5 consecutive epochs
    restore_best_weights=True  # rewind to the best epoch's weights when done
)
```

You pass the callback to `model.fit()` via the `callbacks` argument:

```python
history = model.fit(..., callbacks=[early_stop])
```

With `restore_best_weights=True`, Keras automatically rewinds the model to the weights from the best epoch — so after training completes, `model.predict()` uses those best weights, not the overfitted final-epoch weights.  The training may stop well before the `epochs` limit if no improvement is seen for `patience` epochs.

Let's retrain the baseline MLP with early stopping enabled and see how it behaves.

In [ ]:
# TODO: Recreate the baseline model and compile it again for training with
# early stopping (use your `build_mlp_baseline()` function to create a
# fresh instance of the model.
mlp_es = ...

# TODO: Compile with the same settings as before
mlp_es.compile(
    ...
)

# Define the early-stopping callback
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',   # stop when validation loss stops improving
    patience=5,           # epochs to wait after last improvement to stop
    restore_best_weights=True,
    verbose=1              # print a message when stopping early
)

# Train for up to 50 epochs: But early stopping added via callbacks, so
# fitting should be interrupted when appropriate.
history_es = mlp_es.fit(
    X_train, y_train,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining stopped at epoch {len(history_es.history['loss'])} "
      f"(best weights restored from earlier epoch)")

Plot the loss curve and learning curves again to see how early stopping affected training. Also show the confusion matrix for the early-stopped model and compare it to the baseline.

In [ ]:
plot_history(history_es, 'Baseline MLP with Early Stopping')


**Question 6:** Look at the learning curves for the early-stopping run.

a. At approximately which epoch did validation loss reach its minimum?  How does that compare to the epoch at which training actually stopped?

```
ANSWER HERE
```

**Question 6 (Continued):**

b. The following cell will display both the evaluation report of the initial baseline model and this new early stopping model. How does the test accuracy of the early-stopping model compare to the fixed-15-epoch baseline?  What does this suggest about which epoch's weights you should prefer when making predictions?

```
ANSWER HERE
```

In [ ]:
# Print the original baseline report again for easy comparison with the 
# early-stopped model.
print(baseline_report)

# Generate and print the evaluation report for the early-stopped model
early_file = "baseline_mlp_early_stopping_confusion_matrix.png"
early_stop_report = evaluate_model(mlp_es, X_test, y_test, y_test_raw,
               CLASS_NAMES, title='Baseline MLP (Early Stopping)', 
               filename=early_file)
print(early_stop_report)

**Question 6 (Continued):**

c. The following cell will display the confusion matrix of the original baseline model and the early-stopping model side by side.  Do you see any noticeable differences in which classes are confused with each other?

```
ANSWER HERE
```

In [ ]:
# Display both confusion matrices side by side for comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
# Baseline MLP confusion matrix
baseline_cm = plt.imread(cm_file)
axes[0].imshow(baseline_cm)
axes[0].axis('off')
# Early-stopped MLP confusion matrix
early_cm = plt.imread(early_file)
axes[1].imshow(early_cm)
axes[1].axis('off')

plt.tight_layout()
plt.show()

**Question 6 (Continued):**

d. In your own words, explain why `restore_best_weights=True` is important.  What would happen if you left it as `False`?

```
ANSWER HERE
```

---

## Part C: Adding Dropout

**Dropout** is a regularization technique that randomly sets a fraction of the neurons' output to zero during each training step.  This prevents any single neuron from becoming too important and forces the network to learn more robust, distributed representations — which tends to reduce overfitting.

In Keras, you add a `layers.Dropout(rate)` layer after the layer you want to regularize.  For example:

```python
layers.Dense(256, activation='relu'),
layers.Dropout(0.3),   # randomly zero-out 30% of outputs during training
```

Dropout is **only active during training** — during evaluation/prediction, all neurons are used (but their outputs are scaled appropriately).

**Question 7 (Predict before running):** Before you train the dropout model, write down your prediction: compared to the baseline, do you expect the *training* accuracy to be higher, lower, or about the same?  What about the *validation/test* accuracy?  Explain your reasoning.

```
ANSWER HERE
```

### Step C.1 — Build the Dropout Model

We use the same layer sizes as the baseline but insert a `Dropout(0.3)` layer after each hidden layer.

In [ ]:
def build_dropout_mlp(dropout_rate=0.3):
    """Two hidden-layer MLP with Dropout regularization."""

    # TODO: Add Dropout layers after each hidden layer in the
    # original MLP architecture (reproduced below). Use the `dropout_rate`
    # parameter (passed in above) to set the dropout rate for both layers.
    model = keras.Sequential(name='Dropout_MLP', layers=[
        keras.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(10, activation='softmax'),
    ])
    return model

mlp_dropout = build_dropout_mlp(dropout_rate=0.3)
mlp_dropout.summary()

Notice that Dropout layers have **zero trainable parameters** — all they do is randomly zero out some outputs during each training step — there is nothing to learn or tune.

In [ ]:
# Compile the dropout model with the same settings as before
mlp_dropout.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the dropout model for 15 epochs with a batch size of 128 and a
# 10% validation split, and store the training history in `history_dropout`.
# I've left ourt the early-stopping callback for now, but you can add it
# back in if you want to see how it changes the behavior.
history_dropout = mlp_dropout.fit(
    X_train, y_train,
    batch_size=128,
    epochs=15,
    validation_split=0.1,
    verbose=1
)

### Step C.2 — Evaluate the Dropout Model

In [ ]:
plot_history(history_dropout, 'Dropout MLP — Training History')

**Question 8:** Compare the learning curves and test accuracy of the baseline MLP and the dropout MLP.  Did the results match your prediction in Question 6?  Specifically:
a. How does the *gap* between training and validation accuracy compare?

```
ANSWER HERE
```

**Question 8 (Continued):**

b. Examining the evaluation reports (using the cell below), is the test-set accuracy higher, lower, or about the same?

```
ANSWER HERE
```

In [ ]:
# Print the original baseline report again for easy comparison
print(baseline_report)

# Generate and print the report for this model
dropout_file = "dropout_mlp_confusion_matrix.png"
dropout_report = evaluate_model(mlp_dropout, X_test, y_test, y_test_raw,
               CLASS_NAMES, title='Dropout MLP (rate=0.3)', 
               filename=dropout_file)
print(dropout_report)

**Question 8 (Continued):**


c. Are there visible differences in the confusion matrices generated by the cell below?

```
ANSWER HERE
```

In [ ]:
# Display both confusion matrices side by side for comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
# Baseline MLP confusion matrix
baseline_cm = plt.imread(cm_file)
axes[0].imshow(baseline_cm)
axes[0].axis('off')
# Dropout MLP confusion matrix
drop_cm = plt.imread(dropout_file)
axes[1].imshow(drop_cm)
axes[1].axis('off')

plt.tight_layout()
plt.show()

---

## Part D: Bottleneck Architecture — A Narrow Middle Layer

Now let's explore what happens when we force the network to squeeze all its learned information through a very narrow layer.  We will remove the dropout and instead insert a **bottleneck layer** of only 4 nodes between the two wider hidden layers:

- Hidden layer 1: 256 nodes (ReLU)
- **Bottleneck layer**: 2 nodes (ReLU)   <== new!
- Hidden layer 2: 128 nodes (ReLU)
- Output: 10 nodes (softmax)

**Question 9 (Predict before running):** Before you train this model, write down your prediction: do you think this bottleneck model will perform *better*, *worse*, or *about the same* as the baseline MLP?  Why?  (Think about what it means to try to represent 784-dimensional input information in just 2 numbers.)

```
ANSWER HERE
```

### Step D.1 — Build the Bottleneck Model

The "Bottleneck" architecture just requires you to insert a new `Dense` layer with `bottleneck_size` nodes between the two existing hidden layers.  The code is already set up to take a `bottleneck_size` parameter so you can easily experiment with different sizes later on.

In [ ]:
def build_bottleneck_mlp(bottleneck_size=8):
    """Three hidden-layer MLP with a narrow bottleneck in the middle."""
    model = keras.Sequential(name='Bottleneck_MLP', layers=[
        keras.Input(shape=(784,)),
        # First wide hidden layer
        layers.Dense(256, activation='relu'),
        # TODO: Add narrow bottleneck Dense layer using `bottleneck_size`
        # parameter passed in above as the number of nodes and ReLU
        # activation.
        ...
        # Second wide hidden layer
        layers.Dense(128, activation='relu'),
        # Output layer
        layers.Dense(10, activation='softmax'),
    ])
    return model

mlp_bottleneck = build_bottleneck_mlp(bottleneck_size=2)
mlp_bottleneck.summary()

**Question 10:** Look at the parameter count of the bottleneck model.  Is it larger or smaller than the baseline MLP which had 235,146 parameters (aka weights and biases)?  Which layer accounts for the biggest difference?

```
ANSWER HERE
```

In [ ]:
# Compile the bottleneck model with the same settings as before
mlp_bottleneck.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the bottleneck model for 15 epochs with a batch size of 128 and a
# 10% validation split, and store the training history in `history_bottleneck`.
# (Again, I've left out the early-stopping callback for now, but you can add it
# back in if you want to see how it changes the behavior.)
history_bottleneck = mlp_bottleneck.fit(
    X_train, y_train,
    batch_size=128,
    epochs=15,
    validation_split=0.1,
    verbose=1
)

### Step D.2 — Evaluate the Bottleneck Model

Let's display the information from the evaluation of the bottleneck model side by side with the baseline model for easy comparison.  We will also plot the learning curves and confusion matrix for the bottleneck model to see how it differs from the others.

In [ ]:
plot_history(history_bottleneck, 'Bottleneck MLP — Training History')

In [ ]:
# Display original baseline report again for easy comparison
print(baseline_report)

# Generate and print the evaluation report for the bottleneck model
bottleneck_file = "bottleneck_mlp_confusion_matrix.png"
bottleneck_report = evaluate_model(mlp_bottleneck, X_test, y_test, y_test_raw,
                                   CLASS_NAMES, 
                                   title='Bottleneck MLP (8-node middle layer)',
                                   filename=bottleneck_file
                                   )
print(bottleneck_report)

In [ ]:
# Display both confusion matrices side by side for comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
# Baseline MLP confusion matrix
baseline_cm = plt.imread(cm_file)
axes[0].imshow(baseline_cm)
axes[0].axis('off')
# Bottleneck MLP confusion matrix
bottle_cm = plt.imread(bottleneck_file)
axes[1].imshow(bottle_cm)
axes[1].axis('off')
plt.tight_layout()
plt.show()

**Question 11:** How did the bottleneck model's test accuracy compare to the baseline?  Did the results match your prediction in Question 8?  Look at the confusion matrix — are there particular classes that suffer more from the bottleneck constraint?

```
ANSWER HERE
```

### Step D.3 — Side-by-Side Summary

Run the cell below to print a quick accuracy comparison of all three MLP architectures.

In [ ]:
models = {
    'Baseline MLP (256-128)'       : mlp_baseline,
    'Dropout MLP (256-128, p=0.3)' : mlp_dropout,
    'Bottleneck MLP (256-8-128)'   : mlp_bottleneck,
}

print(f"{'Model':<40} {'Test Accuracy':>14}")
print('-' * 56)
for name, mdl in models.items():
    _, acc = mdl.evaluate(X_test, y_test, verbose=0)
    print(f"{name:<40} {acc*100:>13.2f}%")

**Question 12:** Based on all three experiments, what is the key lesson about **network architecture**?  Why can't you just make the network as narrow as you want to reduce the number of parameters?

```
ANSWER HERE
```

---

## Part E: Convolutional Neural Network (ConvNet)

The MLP discards all spatial structure — it treats the 28×28 image as an unordered list of 784 numbers.  A **Convolutional Neural Network (CNN)** instead processes the image in its 2D form, using small learnable filters that slide across the image to detect local patterns (edges, textures, shapes) regardless of where they appear.

Key CNN building blocks:
- **`Conv2D(filters, kernel_size, activation)`** — applies `filters` learnable 2D kernels of size `kernel_size × kernel_size` to produce a stack of feature maps.
- **`MaxPooling2D(pool_size)`** — downsamples each feature map by taking the maximum value in each non-overlapping `pool_size × pool_size` region.  This reduces spatial dimensions and adds some translation invariance.
- **`Flatten()`** — converts the 3D feature-map volume into a 1D vector so it can be fed into Dense layers.

Note that for a CNN, **the images must be kept in their 2D form** — we reshape them to `(28, 28, 1)` (height × width × channels, where channels=1 for grayscale).

### Step E.1 — Prepare 2D Input

In [ ]:
# Keep images as 28x28 images, normalize, add a channel dimension
# (grayscale = 1 channel).  Normally the array of data would be
# 4D (num_samples, height, width, channels) for a CNN.
X_train_2d = X_train_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0
X_test_2d  = X_test_raw.reshape(-1, 28, 28, 1).astype('float32') / 255.0

print("2D training set shape:", X_train_2d.shape)  # should be (60000, 28, 28, 1)

### Step E.2 — Build the ConvNet

Using the notes from class to guide you, build a simple ConvNet architecture.  We are not building this in a function since we don't plan to reuse it, but feel free to experiment with different architectures if you like.

In [ ]:
cnn_model = keras.Sequential(name='Simple_ConvNet', layers=[
    # Take the 28x28 grayscale image as input (1 channel)
    keras.Input(shape=(28, 28, 1)),

    # TODO: Create the first convolutional filter (using a Conv2D layer)
    # assuming 32 filters and a 3x3 kernel
    ...
    # TODO: Add a 2x2 max pooling layer to reduce spatial dimensions
    ...

    # TODO: Create the second convolutional block consisting of
    # Conv2D layer with 64 filters and a 3x3 kernel, followed by a
    # 2x2 max pooling layer
    ...

    # Flatten and classify using a "hidden" Dense layer with 64 nodes
    # and ReLU activation, followed by the output layer with 10 nodes
    # and softmax activation (as appropriate for classification)
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

cnn_model.summary()

**Question 13:** Compare the number of trainable parameters in the ConvNet to the baseline MLP.  Which has more?  Even so, why might the ConvNet be expected to outperform the MLP on image data?

```
ANSWER HERE
```

### Step E.3 — Train and Evaluate the ConvNet

In [ ]:
# Compile the CNN model with the same settings as before
cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the CNN model for 15 epochs with a batch size of 128 and a
# 10% validation split, and store the training history in `history_cnn`.
history_cnn = cnn_model.fit(
    X_train_2d, y_train,
    batch_size=128,
    epochs=15,
    validation_split=0.1,
    verbose=1
)

Now let's evaluate the ConvNet using the same evaluation code as before.  We will compare its learning curves, test accuracy, and confusion matrix to the baseline MLP.

In [ ]:
plot_history(history_cnn, 'Simple ConvNet — Training History')

In [ ]:
# By this point you have seen the original baseline report enough times,
# so let's just generate and print the report for the CNN model
cnn_file = 'cnn_confusion_matrix.png'
cnn_report = evaluate_model(cnn_model, X_test_2d, y_test, y_test_raw,
               CLASS_NAMES, title='Simple ConvNet', filename=cnn_file)
print(cnn_report)

In [ ]:
# Compare this confusion matrix to the baseline MLP's confusion matrix
# to see if the CNN is doing better on certain classes than the MLP.

# Display both confusion matrices side by side for comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
# Baseline MLP confusion matrix
baseline_cm = plt.imread(cm_file)
axes[0].imshow(baseline_cm)
axes[0].axis('off')
# Bottleneck MLP confusion matrix
cnn_cm = plt.imread(cnn_file)
axes[1].imshow(cnn_cm)
axes[1].axis('off')
plt.tight_layout()
plt.show()

### Step E.4 — Final Summary: All Models

In [ ]:
print(f"{'Model':<40} {'Test Accuracy':>14}")
print('-' * 56)

# MLP models (flat input)
for name, mdl in [
    ('Baseline MLP (256-128)',        mlp_baseline),
    ('Dropout MLP (256-128, p=0.3)',  mlp_dropout),
    ('Bottleneck MLP (256-8-128)',    mlp_bottleneck),
]:
    _, acc = mdl.evaluate(X_test, y_test, verbose=0)
    print(f"{name:<40} {acc*100:>13.2f}%")

# CNN (2D input)
_, acc_cnn = cnn_model.evaluate(X_test_2d, y_test, verbose=0)
print(f"{'Simple ConvNet':<40} {acc_cnn*100:>13.2f}%")

**Question 14:** Compare all four models.  How does the ConvNet compare to the best MLP?  What does this tell you about the importance of choosing an architecture that matches the structure of your data?

```
ANSWER HERE
```

**Question 15 (Reflection):** In this lab you trained four different models on the same dataset without changing the data at all — only the architecture changed.  List at least **three architectural choices** you made (or that were made for you) and briefly describe the effect each had on model performance or training behavior.

```
ANSWER HERE
```

---

## What to Submit

Once you have completed all parts of this lab:

1. Make sure all cells are run and all output is visible (do a final **Runtime → Run all** to confirm).
2. Go to **File → Download → Download .ipynb** to save the notebook to your computer.
3. Submit the `.ipynb` file to the **Lab 9** dropbox on D2L.

You will be graded on:
- **Correct code** (all cells run without error)
- **Written answers** that demonstrate understanding of the concepts
- **Prediction cells** (Questions 6 and 8) filled in *before* running the corresponding experiments